In [1]:
import os
import numpy as np
import pandas as pd

# 出力カラムとゾーンの定義
ZONES = ["TL","TC","TR","ML","MC","MR","BL","BC","BR"]
OUT_COLS = [
    "player_name", "foot_enc", "match_time", "score_diff", 
    "is_shootout", "home_away", "pressure_index", "zone_target", "source"
]

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

# ディレクトリの準備
ensure_dir("data")
print("環境設定が完了しました。")

環境設定が完了しました。


In [2]:
def clip(x, lo, hi):
    return float(np.minimum(np.maximum(x, lo), hi))

def compute_pressure_index(match_time: float, score_diff: int, is_shootout: int = 0) -> float:
    t = clip(match_time / 120.0, 0, 1)
    close = 1.0 if abs(score_diff) <= 1 else 0.6 if abs(score_diff) == 2 else 0.3
    base = 8.0 * t * close
    if int(is_shootout) == 1:
        base = max(base, 6.5)
    return clip(base, 0, 10)

def zone_from_end(end_y: float, end_z: float) -> str:
    y, z = end_y, end_z
    col = "L" if y < 38.5 else "C" if y < 42.0 else "R"
    row = "B" if z < 0.9 else "M" if z < 1.8 else "T"
    zone = f"{row}{col}"
    return zone if zone in ZONES else "MC"

# ロジックの検証
test_zone = zone_from_end(37.0, 0.5) # 左下を期待
test_pressure = compute_pressure_index(115, 0, 1) # 高圧力を期待
print(f"検証結果 - ゾーン: {test_zone}, 圧力指数: {test_pressure:.2f}")

検証結果 - ゾーン: BL, 圧力指数: 7.67


In [4]:
def sample_match_time(n: int, rng: np.random.Generator) -> np.ndarray:
    u = rng.random(n)
    t = np.empty(n)
    t[u < 0.65] = rng.integers(46, 91, size=(u < 0.65).sum())
    t[(u >= 0.65) & (u < 0.90)] = rng.integers(1, 46, size=((u >= 0.65) & (u < 0.90)).sum())
    t[u >= 0.90] = rng.integers(91, 121, size=(u >= 0.90).sum())
    return t.astype(int)

def sample_score_diff(n: int, rng: np.random.Generator) -> np.ndarray:
    values, probs = np.array([-3, -2, -1, 0, 1, 2, 3]), np.array([0.03, 0.07, 0.20, 0.40, 0.20, 0.07, 0.03])
    return rng.choice(values, size=n, p=probs).astype(int)

def load_real(real_csv_path: str, seed: int = 42) -> pd.DataFrame:
    if not os.path.exists(real_csv_path):
        raise FileNotFoundError(f"ファイルが見つかりません: {real_csv_path}")
    
    df = pd.read_csv(real_csv_path)
    rng = np.random.default_rng(seed)
    
    # 利き足のエンコード (Right=1, Left=0)
    df["foot_enc"] = np.where(df["body_part"].str.contains("Right", case=False, na=False), 1, 0)
    
    # オプション項目の生成
    if "match_time" not in df.columns: df["match_time"] = sample_match_time(len(df), rng)
    if "score_diff" not in df.columns: df["score_diff"] = sample_score_diff(len(df), rng)
    if "is_shootout" not in df.columns: df["is_shootout"] = 0
    if "home_away" not in df.columns: df["home_away"] = rng.integers(0, 2, size=len(df))
    
    df["pressure_index"] = [round(compute_pressure_index(t, sd, so), 2) 
                            for t, sd, so in zip(df["match_time"], df["score_diff"], df["is_shootout"])]
    df["zone_target"] = [zone_from_end(y, z) for y, z in zip(df["end_y"], df["end_z"])]
    
    res = df[["player_name", "foot_enc", "match_time", "score_diff", "is_shootout", "home_away", "pressure_index", "zone_target"]].copy()
    res["source"] = "real"
    return res

# 実行
real_df = load_real("real_penalties.csv")
print(real_df.head())

                   player_name  foot_enc  match_time  score_diff  is_shootout  \
0         Victor Okoh Boniface         1          42           0            0   
1                Florian Wirtz         1          51           2            0   
2  Exequiel Alejandro Palacios         1           7           3            0   
3         Victor Okoh Boniface         1          15          -2            0   
4         Victor Okoh Boniface         1          62           0            0   

   home_away  pressure_index zone_target source  
0          1            2.80          MR   real  
1          0            2.04          ML   real  
2          1            0.14          BR   real  
3          0            0.60          TC   real  
4          1            4.13          BC   real  


In [5]:
def generate_synthetic(real_df: pd.DataFrame, n_samples: int, seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = pd.DataFrame()
    out["player_name"] = rng.choice(real_df["player_name"].values, size=n_samples, replace=True)
    out["foot_enc"] = rng.choice([0, 1], size=n_samples, p=[0.25, 0.75])
    out["match_time"] = sample_match_time(n_samples, rng)
    out["score_diff"] = sample_score_diff(n_samples, rng)
    out["is_shootout"] = rng.choice([0, 1], size=n_samples, p=[0.92, 0.08])
    out["home_away"] = rng.integers(0, 2, size=n_samples)
    out["pressure_index"] = [round(compute_pressure_index(t, sd, so), 2) 
                            for t, sd, so in zip(out["match_time"], out["score_diff"], out["is_shootout"])]
    
    # 選手の傾向に基づいたゾーンサンプリング
    counts = real_df.groupby(["player_name", "zone_target"]).size().unstack(fill_value=0).reindex(columns=ZONES, fill_value=0)
    probs = (counts + 1.0).div((counts + 1.0).sum(axis=1), axis=0)
    
    out["zone_target"] = [rng.choice(ZONES, p=(probs.loc[p].values if p in probs.index else np.ones(len(ZONES))/len(ZONES))) 
                          for p in out["player_name"]]
    out["source"] = "synthetic"
    return out

print("合成データ生成ロジックが定義されました。")

合成データ生成ロジックが定義されました。


In [7]:
def build_hybrid(real_path, out_path, multiplier=1.0):
    real_df = load_real(real_path)
    synth_df = generate_synthetic(real_df, int(len(real_df) * multiplier))
    
    hybrid = pd.concat([real_df, synth_df], ignore_index=True)[OUT_COLS]
    hybrid.to_csv(out_path, index=False)
    return hybrid

# 最終実行と評価
try:
    final_df = build_hybrid("real_penalties.csv", "hybrid_penalties.csv", multiplier=1.0)
    print("有効性評価:")
    print(f"- 総サンプル数: {len(final_df)}")
    print(f"- データソース別内訳:\n{final_df['source'].value_counts()}")
    print(f"- ゾーン別分布:\n{final_df['zone_target'].value_counts(normalize=True).head()}")
    print("\nデータの保存が完了しました。")
except Exception as e:
    print(f"実行エラー: {e}")

有効性評価:
- 総サンプル数: 2722
- データソース別内訳:
source
real         1361
synthetic    1361
Name: count, dtype: int64
- ゾーン別分布:
zone_target
BL    0.214916
BR    0.177076
ML    0.126378
MR    0.107641
BC    0.080456
Name: proportion, dtype: float64

データの保存が完了しました。
